# Решения: set, dict и частоты

**Для преподавателя.** Полный эталон к `lesson.ipynb` и `homework.ipynb`; ученикам до сдачи не показывать.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def find_orders_csv():
    for path in (
        Path("orders_slim.csv"),
        Path("../orders_slim.csv"),
        Path("../../data/orders_slim.csv"),
        Path("../data/orders_slim.csv"),
        Path("../../../data/orders_slim.csv"),
    ):
        if path.exists():
            return path.resolve()
    return (
        "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/"
        "modules/08_08_logistics_clustering/data/orders_slim.csv"
    )


CSV_PATH = find_orders_csv()
DATE_COLUMNS = [
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
    "order_delivered_customer_date",
]
df = pd.read_csv(CSV_PATH, parse_dates=DATE_COLUMNS)
assert len(df) > 0
assert df["order_id"].notna().all()
print(f"Загружено заказов: {len(df)}")


## Урок. 1–2. Set и membership

In [ ]:
unique_orders = set(df["order_id"])
duplicate_count = len(df) - len(unique_orders)
known = unique_orders
probes = df["order_id"].head(5).tolist() + ["missing_order"]
found = [oid in known for oid in probes]
assert found[-1] is False


## Урок. 3. Частоты customer_state

In [ ]:
customer_counts = {}
for state in df["customer_state"]:
    customer_counts[state] = customer_counts.get(state, 0) + 1
assert sum(customer_counts.values()) == len(df)


## Урок. 4–5. Late count и rate

In [ ]:
customer_late = {state: 0 for state in customer_counts}
for row in df[["customer_state", "is_late"]].itertuples(index=False):
    customer_late[row.customer_state] += int(row.is_late)
customer_rate = {state: customer_late[state] / customer_counts[state] for state in customer_counts}
assert sum(customer_late.values()) == int(df["is_late"].sum())


## Урок. 6. Статистика seller_state

In [ ]:
seller_stats = {}
for row in df[["seller_state", "is_late"]].itertuples(index=False):
    bucket = seller_stats.setdefault(row.seller_state, {"late": 0, "total": 0})
    bucket["total"] += 1
    bucket["late"] += int(row.is_late)
assert sum(v["total"] for v in seller_stats.values()) == len(df)


## Урок. 7–8. Рейтинг и смысл

In [ ]:
late_ranking = sorted(customer_late.items(), key=lambda item: item[1], reverse=True)
leader_count = max(customer_late, key=customer_late.get)
leader_rate = max(customer_rate, key=customer_rate.get)
RATE_NOTE = "Count показывает объём late-заказов и зависит от размера региона. Rate отвечает на вопрос о доле проблем внутри региона; без total маленький сегмент может выглядеть главным риском случайно."
assert len(RATE_NOTE) >= 100


## Урок. 9. Универсальная функция

In [ ]:
def late_stats(frame, segment_column):
    result = {}
    for row in frame[[segment_column, "is_late"]].itertuples(index=False, name=None):
        segment, is_late = row
        bucket = result.setdefault(segment, {"late": 0, "total": 0, "rate": 0.0})
        bucket["total"] += 1
        bucket["late"] += int(is_late)
    for bucket in result.values():
        bucket["rate"] = bucket["late"] / bucket["total"]
    return result

stats = late_stats(df, "seller_state")
assert sum(v["total"] for v in stats.values()) == len(df)


## ДЗ. A1–A3

In [ ]:
seller_ids = set(df["seller_id"])
seller_late = {}
for row in df.loc[df["is_late"].eq(1), ["seller_id"]].itertuples(index=False):
    seller_late[row.seller_id] = seller_late.get(row.seller_id, 0) + 1
top5_sellers = sorted(seller_late.items(), key=lambda item: item[1], reverse=True)[:5]
assert set(seller_late) <= seller_ids


## ДЗ. Challenge

In [ ]:
rows = [{"segment": segment, **values} for segment, values in late_stats(df, "seller_state").items()]
state_table = pd.DataFrame(rows).sort_values(["rate", "total"], ascending=False)
FREQUENCY_NOTE = "Late count нужен для оценки объёма работы, late rate — для сравнения качества сегментов. Оба числа читаются вместе с total: высокая доля на двух заказах слабее как основание для решения, чем устойчивая доля на сотнях."
assert len(FREQUENCY_NOTE) >= 180
